# Ethiopia Climate EDA

Week 0 Task 2 notebook for the African Climate Trend Analysis challenge.

This notebook covers:
- data loading and date parsing
- missing-value handling and duplicate removal
- outlier detection
- time series, correlation, and distribution analysis
- cleaned data export

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import zscore

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
def resolve_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find any of: {candidates}")


raw_path = resolve_path("data/ethiopia.csv", "../data/ethiopia.csv")
clean_path = raw_path.with_name("ethiopia_clean.csv")
raw_path, clean_path

## Data Loading And Date Parsing

In [ ]:
df = pd.read_csv(raw_path)
df["Country"] = "Ethiopia"
df["date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["Month"] = df["date"].dt.month
df = df.sort_values("date").reset_index(drop=True)

print(f"Rows, columns: {df.shape}")
df.head()

## Replace NASA Sentinel Values

In [ ]:
df = df.replace(-999, np.nan)
df.head()

## Duplicates

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

## Summary Statistics

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
summary_stats = df[numeric_cols].describe().T
summary_stats

Write a short interpretation here after running the summary table. Focus on the average temperature, precipitation spread, and any unusually wide ranges.

## Missing-Value Report

In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct,
})
missing_report

In [ ]:
missing_over_5 = missing_report[missing_report["missing_pct"] > 5]
missing_over_5

List any columns above 5% missingness here and explain what that may mean for reliability or interpretation.

## Outlier Detection

In [ ]:
outlier_cols = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "PRECTOTCORR",
    "RH2M",
    "WS2M",
    "WS2M_MAX",
]

zscore_frame = df[outlier_cols].apply(lambda col: zscore(col, nan_policy="omit"))
outlier_mask = zscore_frame.abs().gt(3).any(axis=1)
outlier_count = int(outlier_mask.sum())

print(f"Rows flagged as outliers: {outlier_count}")
df.loc[outlier_mask, ["date"] + outlier_cols].head()

Document your outlier decision here. In climate data, it is usually better to retain real extreme events unless you find obvious measurement errors.

## Missing-Value Cleaning

In [ ]:
weather_fill_cols = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "PRECTOTCORR",
    "RH2M",
    "WS2M",
    "WS2M_MAX",
    "PS",
    "QV2M",
]

row_missing_pct = df.isna().mean(axis=1) * 100
rows_over_30 = int((row_missing_pct > 30).sum())
print(f"Rows with more than 30% missing values: {rows_over_30}")

df_clean = df.loc[row_missing_pct <= 30].copy()
df_clean[weather_fill_cols] = df_clean[weather_fill_cols].ffill()

print(f"Cleaned shape: {df_clean.shape}")
df_clean.isna().sum()

Document the cleaning choice here: rows above 30% missing were dropped and the remaining weather fields were forward-filled.

## Export Cleaned Data

In [ ]:
df_clean.to_csv(clean_path, index=False)
print(f"Saved cleaned data to {clean_path}")

## Time Series Analysis

In [ ]:
monthly = (
    df_clean.set_index("date")
    .resample("MS")
    .agg({"T2M": "mean", "PRECTOTCORR": "sum"})
    .reset_index()
)

warmest = monthly.loc[monthly["T2M"].idxmax()]
coolest = monthly.loc[monthly["T2M"].idxmin()]
rainiest = monthly.loc[monthly["PRECTOTCORR"].idxmax()]

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly, x="date", y="T2M", ax=ax, color="firebrick")
ax.scatter(warmest["date"], warmest["T2M"], color="darkred", s=70)
ax.scatter(coolest["date"], coolest["T2M"], color="navy", s=70)
ax.annotate("Warmest month", (warmest["date"], warmest["T2M"]), xytext=(8, 8), textcoords="offset points")
ax.annotate("Coolest month", (coolest["date"], coolest["T2M"]), xytext=(8, -14), textcoords="offset points")
ax.set_title("Monthly Average T2M in Ethiopia (2015-2026)")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=monthly, x=monthly["date"].dt.strftime("%Y-%m"), y="PRECTOTCORR", ax=ax, color="teal")
peak_index = monthly["PRECTOTCORR"].idxmax()
ax.annotate(
    "Peak rainy month",
    (peak_index, rainiest["PRECTOTCORR"]),
    xytext=(0, 10),
    textcoords="offset points",
    ha="center",
)
ax.set_title("Monthly Total PRECTOTCORR in Ethiopia (2015-2026)")
ax.set_xlabel("Month")
ax.set_ylabel("Precipitation (mm)")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

Comment on visible trends or anomalies here. Mention warming periods, cooler intervals, or unusually wet months.

## Correlation And Relationships

In [ ]:
corr = df_clean[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_clean, x="T2M", y="RH2M", ax=axes[0], alpha=0.6)
axes[0].set_title("T2M vs RH2M")

sns.scatterplot(data=df_clean, x="T2M_RANGE", y="WS2M", ax=axes[1], alpha=0.6)
axes[1].set_title("T2M_RANGE vs WS2M")
plt.tight_layout()
plt.show()

In [ ]:
strongest_corr = (
    corr.where(~np.eye(corr.shape[0], dtype=bool))
    .stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .drop_duplicates()
)
strongest_corr.head(3)

Interpret the three strongest correlations here. Explain whether they make physical sense in the context of climate variables.

## Distribution Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_clean["PRECTOTCORR"].dropna(), bins=40, ax=ax, color="slateblue")
ax.set_title("Distribution of PRECTOTCORR")
ax.set_xlabel("Precipitation (mm/day)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
bubble_size = df_clean["PRECTOTCORR"].fillna(0).clip(lower=0) * 8 + 10
scatter = ax.scatter(
    df_clean["T2M"],
    df_clean["RH2M"],
    s=bubble_size,
    c=df_clean["PRECTOTCORR"],
    cmap="viridis",
    alpha=0.45,
)
ax.set_title("T2M vs RH2M with PRECTOTCORR Bubble Size")
ax.set_xlabel("T2M (C)")
ax.set_ylabel("RH2M (%)")
plt.colorbar(scatter, ax=ax, label="PRECTOTCORR")
plt.tight_layout()
plt.show()

Comment on the precipitation distribution here. If the histogram is heavily right-skewed, note that a log-scaled view may be useful.